# 1. FAISS (Understand the Fundamentals)

In [1]:
#1 — Imports
from pathlib import Path
import pickle
import os
import faiss
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings



In [2]:
#2 — Load Environment
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [3]:
#3 — Paths
PROJECT_ROOT = Path("..").resolve()

ARTIFACTS = PROJECT_ROOT / "artifacts"

INPUT_FILE = ARTIFACTS / "vector_records.pkl"

FAISS_INDEX = ARTIFACTS / "faiss_index.index"

METADATA_FILE = ARTIFACTS / "faiss_metadata.pkl"

In [4]:
#4 — Load Vector Records
with open(INPUT_FILE,"rb") as f:
    vector_records = pickle.load(f)

print(len(vector_records))


5073


In [5]:
#5 — Embedding Dimension
dimension = len(vector_records[0]["values"])
print(dimension)

768


In [6]:
#7 — Build NumPy Matrix
vectors = np.array(
    [record["values"] for record in vector_records],
    dtype=np.float32
)
vectors.shape

(5073, 768)

In [7]:
vectors

array([[-0.01929239,  0.03620932, -0.00546486, ...,  0.00307867,
         0.04371994, -0.00934114],
       [-0.01108941,  0.01080698, -0.00160594, ...,  0.01416938,
         0.03521179, -0.00063615],
       [-0.02257641,  0.01768476,  0.00639293, ...,  0.0144376 ,
         0.03531457,  0.00028342],
       ...,
       [-0.01495351,  0.00337279, -0.00599988, ...,  0.01749548,
         0.0052838 , -0.00760762],
       [-0.02694327,  0.0087275 , -0.00428525, ...,  0.01165542,
        -0.00052106, -0.01219714],
       [-0.01428397,  0.01720113, -0.01153967, ...,  0.00071364,
         0.01065319, -0.01714497]], shape=(5073, 768), dtype=float32)

In [8]:
#8 — Create FAISS Index
index = faiss.IndexFlatL2(dimension)
index

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x0000024D16D38870> >

In [9]:
#9 — Add Vectors - 
index.add(vectors)
print(index.ntotal)

5073


In [10]:
#10 — Save Index - faiss_index.index
faiss.write_index(
    index,
    str(FAISS_INDEX)
)

In [11]:
#11 — Save Metadata
metadata = []

for record in vector_records:

    metadata.append({"id": record["id"],
                    "metadata": record["metadata"],
                    "text": record["text"]})

with open(METADATA_FILE,"wb") as f:

    pickle.dump(metadata,f)

In [12]:
#12 — Reload Everything
index = faiss.read_index(str(FAISS_INDEX))

with open(METADATA_FILE,"rb") as f:

    metadata = pickle.load(f)

In [13]:
#13 — Build Query Embedding
embedding_model = GoogleGenerativeAIEmbeddings(

    model="models/gemini-embedding-001",
    output_dimensionality=768

)

In [14]:
question = "I notice that people I assigned the test in October of 2025 have not received new tests. How long do the tests stay active in the system."
query_vector = embedding_model.embed_query(question)
type(query_vector)

list

In [15]:
#14 — Convert to NumPy
query_vector = np.array([query_vector], dtype=np.float32)
print(type(query_vector))
print(query_vector.shape)

<class 'numpy.ndarray'>
(1, 768)


In [16]:
#15 — Search
TOP_K = 5
distances, indices = index.search(query_vector, TOP_K)

In [17]:
distances

array([[0.20559932, 0.20884699, 0.21033525, 0.21269572, 0.2142015 ]],
      dtype=float32)

In [18]:
indices[0]

array([4200, 4201, 1931, 4269, 4798])

In [22]:
metadata[0]['text']


"How do I learn more about Amazon and Anthropic’s strategic collaboration?\n_Last updated: 2026-03-16T20:39:36Z_\nLearn more by viewing this press release.\nRelated Articles\nHow do I get access to Claude in Amazon Bedrock?\nHow can I learn more about Claude API pricing?\nHow do I pay for my Claude API usage?\nHow can I access the personal information that Anthropic has on my account?\nWhere can I learn more about Anthropic's Privacy practices?"

In [19]:
#16 — Display Results
for rank, idx in enumerate(indices[0], start=1):

    print("="*80)

    print(f"Rank : {rank}")

    print()

    print(metadata[idx]["metadata"])

    print()

    print(metadata[idx]["text"][:1000])

Rank : 1

{'document_id': '89d4e6f682124ae2d9621219dadfbdc5ec2bc5f3cbc2c1dcaa23e08cd276348c', 'document_hash': 'f2b15bdb75f06683e57dddd3bd4501d4328f3cec02aa52b2de524313b4815665', 'company': 'hackerrank', 'product_area': 'screen', 'title': '"Invite Candidates to a Test Prerequisites Inviting candidates to a test"', 'url': '"https://support.hackerrank.com/articles/6027855406-inviting-candidates-to-a-test"', 'file_name': '6027855406-inviting-candidates-to-a-test.md', 'source_path': 'hackerrank\\screen\\invite-candidates\\6027855406-inviting-candidates-to-a-test.md', 'extension': '.md', 'source_type': 'markdown', 'schema_version': '1.0', 'indexed_at': '2026-07-07T20:51:18.248083+00:00', 'last_modified': '2026-07-02T21:53:34.063812+00:00', 'chunk_number': 3, 'chunk_id': '9482de42f0d579ead048c64d362a400f44e6f8e54eac9e80e0a061b1167b4278', 'content_hash': '3af650912dd4f15ebaf53ef77eb2e24ea0b2e44a93e07ff607af85f86e23f146', 'chunk_size': 964, 'word_count': 153}

<!-- -->
(Optional) Customize the

In [ ]:
#17 — Show Distances
for d in distances[0]:
    print(d)

0.15396288
0.15995751
0.17567077
0.17645867
0.17992589


In [ ]:
#18 — Summary
print("="*60)

print("FAISS COMPLETE")

print("="*60)

print(f"Vectors Indexed : {index.ntotal}")

print(f"Dimensions : {dimension}")

print(f"Index File : {FAISS_INDEX}")


FAISS COMPLETE
Vectors Indexed : 5073
Dimensions : 768
Index File : C:\Users\cheru\OneDrive\Desktop\GitHub_Projects\SupportSphere-AI\artifacts\faiss_index.index


## Creating Multiple Indexes

In [ ]:
#1 — Imports
from pathlib import Path
import pickle
import os
import faiss
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

#2 — Load Environment
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

#3 — Paths
PROJECT_ROOT = Path("..").resolve()

ARTIFACTS = PROJECT_ROOT / "artifacts"

FAISS_INDEX = ARTIFACTS / "faiss_index.index"

METADATA_FILE = ARTIFACTS / "faiss_metadata.pkl"

In [ ]:
#12 — Reload Everything
index = faiss.read_index(str(FAISS_INDEX))

with open(METADATA_FILE,"rb") as f:
    metadata = pickle.load(f)

In [ ]:
#Create a dictionary
indexes = {}

metadata_store = {}

In [ ]:
#4 — Load Vector Records
with open(INPUT_FILE,"rb") as f:
    vector_records = pickle.load(f)

print(len(vector_records))

#7 — Build NumPy Matrix
vectors = np.array(
    [record["values"] for record in vector_records],
    dtype=np.float32
)
vectors.shape

5073


(5073, 768)

In [ ]:
#Global index
indexes["global"] = faiss.IndexFlatL2(dimension)

indexes["global"].add(vectors)

metadata_store["global"] = metadata

In [ ]:
#Now build company indexes.
from collections import defaultdict

company_vectors = defaultdict(list)

company_metadata = defaultdict(list)

for record in vector_records:

    company = record["metadata"]["company"]

    company_vectors[company].append(
        record["values"]
    )

    company_metadata[company].append(
        {
            "id": record["id"],
            "metadata": record["metadata"],
            "text": record["text"]
        }
    )


In [ ]:
#Create FAISS indexes.
for company in company_vectors:

    matrix = np.array(

        company_vectors[company],

        dtype=np.float32

    )

    index = faiss.IndexFlatL2(
        dimension
    )

    index.add(matrix)

    indexes[company] = index

    metadata_store[company] = company_metadata[
        company
    ]

In [ ]:
#Save them
ARTIFACTS = Path("../artifacts")

for name, index in indexes.items():

    faiss.write_index(

        index,

        str(
            ARTIFACTS /
            f"{name}.index"
        )
    )

with open(
    ARTIFACTS/"metadata_store.pkl",
    "wb"
) as f:

    pickle.dump(metadata_store, f)